# Project Delphi (Merlin) 🧙

## 05 - Merlin Console: Colab Deployment Prototype (SentenceTransformer + XGBoost)

### Overview:

In this notebook, we’ll turn Merlin from a collection of training notebooks into a **deployable hybrid model inside Colab**. Instead of retraining anything, we’ll load the artifacts we built in:

- `01_data_preparation` (cleaned dataset: titles + metadata + targets)  
- `02_title_embeddings` (SentenceTransformer embeddings for video titles)  
- `03a_model_training_views_xgboost` (XGBoost model for log-views)  
- `03b_model_training_subscribers_xgboost` (XGBoost model for scaled subscriber change)

The goal is to stack our **SentenceTransformer + dual XGBoost regressors** into a **sinlge resuable prediction function**, then wrap it in a simple “Merlin console” so we can type a proposed episode (title, length, day, month) and instantly see predicted **lifetime views** and **subscriber change** -- all within Colab.

No Streamlit, no retraining. Just a clean inference pipeline we can demo and reuse.

---

### The Plan:
1. **Load artifacts**  
   Import the trained models (`xgb_views_model.pkl`, `xgb_subs_model.pkl`), the RobustScaler for subscribers, and the feature schemas for views and subs.

2. **Rebuild the feature pipeline**  
   Implement a helper that takes raw inputs *(title, episode length in minutes, day of week, month)* and:
   - maps day and month to the same integer encodings used in `01_data_preparation`,  
   - generates a 384-dim title embedding with `all-MiniLM-L6-v2`,  
   - assembles a feature row aligned with the training feature lists.

3. **Stack the models in a single function**  
   Write `merlin_predict(title, duration_min, day_name, month_name)` that:
   - calls the feature builder,  
   - runs the views model (log-space to `expm1` back to views),  
   - runs the subs model (scaled to inverse-transform back to subscribers),  
   - returns interpretable integer predictions.

4. **Build the “Merlin Console” UI**  
   Add a small text-based loop (`merlin_console()`) that prompts for title, length, day, and month, then prints Merlin’s predictions for views and subs in a friendly format.

5. **(Optional) Save a one-click example**  
   Include a pre-filled example cell that calls `merlin_predict(...)` for a sample episode so the notebook runs top-to-bottom as a quick demo.

---

### Why This Matters
- **Proves end-to-end deployment thinking**  
  This notebook shows how to go from *raw idea* to *training* to **reusable inference function**.

- **De-risks Streamlit / production**  
  By having a stable Colab inference path, we can debug the model stack in a controlled environment before worrying about web apps, containers, or Streamlit Cloud.

- **Creates a demo-ready Merlin v0**  
  With a single function call (or a quick console interaction), we can explore what-if scenarios for new episodes — a concrete artifact to share in your portfolio, in interviews, or with the show’s host.

- **Keeps inference logic in one place**  
  Centralizing the feature building and prediction code here makes it the “source of truth” for any future deployment (Streamlit, API, CLI, etc.).

## Load the artifacts

In this step, we mount Google Drive and load every saved component from the 03a/03b training notebooks -- including both XGBoost models, the subscriber scaler, the feature schemas, and the SentenceTransformer embedder. This way, Merlin can make predictions using the *exact same setup* the models were trained with.

In [1]:
# Mount Drive
from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path

# Set the base directory
BASE_DIR = Path("/content/drive/MyDrive/Colab Notebooks/project_delphi")
MODELS = BASE_DIR / "models"

print("BASE_DIR exists:", BASE_DIR.exists())
print("MODELS exists:", MODELS.exists())

print("\nModels folder contents:")
!ls -lh "/content/drive/MyDrive/Colab Notebooks/project_delphi/models"


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
BASE_DIR exists: True
MODELS exists: True

Models folder contents:
total 3.1M
-rw------- 1 root root 4.9K Nov 25 00:40 features_subs.json
-rw------- 1 root root 4.9K Nov 25 01:37 features_views.json
-rw------- 1 root root  405 Nov 12 02:42 subs_model_meta.json
-rw------- 1 root root  402 Nov 12 03:34 views_model_meta.json
-rw------- 1 root root 1.2M Nov 25 00:39 xgb_subs_booster.json
-rw------- 1 root root 916K Nov 25 00:39 xgb_subs_model.pkl
-rw------- 1 root root  511 Nov 25 00:40 xgb_subs_target_scaler.pkl
-rw------- 1 root root 263K Nov 25 01:37 xgb_views_booster.json
-rw------- 1 root root 402K Nov 25 04:33 xgb_views_booster.ubjson
-rw------- 1 root root 406K Nov 25 01:37 xgb_views_model.pkl


In [6]:
# Load the models, schemas, and the embedder into memory
# Start with the imports
import json
import joblib
import numpy as np
import pandas as pd
import xgboost as xgb
from sentence_transformers import SentenceTransformer

# XGBoost views model -- canonical Merlin model = raw booster
booster_views = xgb.Booster()
booster_views.load_model(MODELS / "xgb_views_booster.json")

# XGBoost subs model -- sklearn-style regressor (from 03b)
subs_model = joblib.load(MODELS / "xgb_subs_model.pkl")

# RobustScaler for subscriber target -- from 03b
subs_scaler = joblib.load(MODELS / "xgb_subs_target_scaler.pkl")

# Feature schemas
with open(MODELS / "features_views.json") as f:
    views_schema = json.load(f)["feature_cols"]

with open(MODELS / "features_subs.json") as f:
    subs_schema = json.load(f)["feature_cols"]

print(f"Loaded {len(views_schema)} view features.")
print(f"Loaded {len(subs_schema)} subs features.")

# SentenceTransformer -- same model as 02 / 03a / 03b / 04
embedder = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")

# For the console UI / encoding
day_options = ["Monday", "Tuesday", "Wednesday", "Thursday", "Friday", "Saturday", "Sunday"]
month_options = [
    "January","February","March","April","May","June",
    "July","August","September","October","November","December"
]

# Quick check
print("SentenceTransformer loaded. Merlin is armed ⚔️")

Loaded 387 view features.
Loaded 387 subs features.


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


SentenceTransformer loaded. Merlin is armed ⚔️


## Build the feature builder

Here we recreate Merlin’s full preprocessing pipeline: embedding the title, encoding time-based features, assembling all numeric + embedding features into a single row, and making sure they exactly match training-time column order for both the views and subscribers models.

In [7]:
# Build the feature matrices for views + subs
def build_feature_frames(title: str,
                         duration_min: float,
                         day_name: str,
                         month_name: str):
    """
    Turn raw inputs into X_views and X_subs with the same columns/order
    used during training in 03a / 03b.
    """

    if day_name not in day_options:
        raise ValueError(f"day_name must be one of {day_options}, got {day_name!r}")
    if month_name not in month_options:
        raise ValueError(f"month_name must be one of {month_options}, got {month_name!r}")

    # Encode day/month as intergers -- same convention as 01_data_preparation
    day_idx = day_options.index(day_name)      # 0–6
    month_idx = month_options.index(month_name)  # 0–11

    # Title embedding (same model + settings as training: raw, not normalized)
    title_vec = embedder.encode([title])[0]    # shape (384,)

     # Base numeric / categorical features
    feat = {
        "duration_seconds": float(duration_min) * 60.0,
        "day_of_week": int(day_idx),
        "month": int(month_idx),
    }

    # Add embedding dimensions
    for i, value in enumerate(title_vec):
        feat[f"embed_{i}"] = float(value)

    # Create a single-row DataFrame with all features
    df_input = pd.DataFrame([feat])

    # Match with training schemas for each model
    X_views = df_input.reindex(columns=views_schema, fill_value=0.0)
    X_subs  = df_input.reindex(columns=subs_schema,  fill_value=0.0)

    return X_views, X_subs

In [8]:
# Quick smoke test for feature builder
Xv, Xs = build_feature_frames(
    title="49ers lose to Rams",
    duration_min=30,
    day_name="Monday",
    month_name="January"
)

# Display the shapes and columns
print("X_views shape:", Xv.shape)
print("X_subs shape:", Xs.shape)
print("First 5 columns (views):", list(Xv.columns[:5]))

X_views shape: (1, 387)
X_subs shape: (1, 387)
First 5 columns (views): ['duration_seconds', 'day_of_week', 'month', 'embed_0', 'embed_1']


## Stack the models to build the prediction functions

Now that we can construct the model-ready feature matrices, we define two small functions: one to predict lifetime views (using the log to expm1 reverse transform from 03a), and one to predict subscriber change (using the inverse RobustScaler transform from 03b). These functions apply the trained models exactly as they were used during evaluation.

In [9]:
# Define the prediction functions for views and subscribers
# First, views:
def predict_views(X_views):
    """
    Predict lifetime views using the trained XGBoost booster,
    matching the 03a booster-based predictions.
    """
    dmat = xgb.DMatrix(X_views)
    y_log = booster_views.predict(dmat)[0]   # Predictions in log-space
    y = np.expm1(y_log)                      # Trasnform back to views
    y = max(0, y)
    return int(round(y))

# Now, subscribers:
def predict_subs(X_subs):
    """
    Predict subscriber change using the trained XGBoost model.
    Applies the inverse RobustScaler transform from 03b.
    """
    y_scaled = subs_model.predict(X_subs)[0]     # This is a scaled-space prediction
    y = subs_scaler.inverse_transform([[y_scaled]])
    return int(round(y[0][0]))

In [13]:
# A quick check to make sure both functions are working
# Quick prediction test
test_views = predict_views(Xv)
test_subs = predict_subs(Xs)

# Display the test results
print("Views:", test_views)
print("Subs:", test_subs)

Views: 1803
Subs: 4


## Build the "Merlin Console" UI

This function wraps feature engineering and both XGBoost models into one clean call, letting us generate Merlin predictions from a simple set of inputs.

In [14]:
# Build one clean function to run the whole stack
def merlin_predict(title, duration_min, day_name, month_name):
    """
    Run the full Merlin pipeline for a single hypothetical episode.
    Returns (views, subs).
    """
    # 1) Build model-ready feature frames
    X_views, X_subs = build_feature_frames(
        title=title,
        duration_min=duration_min,
        day_name=day_name,
        month_name=month_name,
    )

    # 2) Predict with the stacked models
    views = predict_views(X_views)
    subs  = predict_subs(X_subs)

    return views, subs

In [15]:
# Check the results with the same hypothetical episode
# we have been using
merlin_predict("49ers lost to Rams", 30, "Monday", "January")

(1857, 5)

This lightweight text interface lets us type in hypothetical episode scenarios -- title, length, day or week, and month -- and instantly ask Merlin for predicted views and subscriber impact.

In [39]:
# Define the merlin_console function
def run_merlin_console():
    print("MERLIN 🧙")
    print("The what-if engine for YouTube creators.")
    print("Type 'quit' as the title to exit at any time.\n")

    while True:
        title = input("Video title: ").strip()
        if title.lower() in {"q", "quit", "exit"}:
            print("Goodbye!")
            break

        # Episode length
        try:
            duration_min = float(input("Episode length in minutes (e.g. 30): "))
        except ValueError:
            print("⚠️ Invalid duration, try again.\n")
            continue

        # Day of week
        print(f"Day options: {', '.join(day_options)}")
        day_name = input("Day of week: ").strip()
        if day_name not in day_options:
            print("⚠️ Invalid day, try again.\n")
            continue

        # Month
        print(f"Month options: {', '.join(month_options)}")
        month_name = input("Month: ").strip()
        if month_name not in month_options:
            print("⚠️ Invalid month, try again.\n")
            continue

        views, subs = merlin_predict(title, duration_min, day_name, month_name)
        print(f"\nMERLIN SAYS → ~{views:,} lifetime views, {subs:+d} subscribers\n")



And now, to consult Merlin...

In [44]:
run_merlin_console()

MERLIN 🧙
The what-if engine for YouTube creators.
Type 'quit' as the title to exit at any time.

Video title: Warriors win the NBA Finals AGAIN in thriller
Episode length in minutes (e.g. 30): 57
Day options: Monday, Tuesday, Wednesday, Thursday, Friday, Saturday, Sunday
Day of week: Tuesday
Month options: January, February, March, April, May, June, July, August, September, October, November, December
Month: June

MERLIN SAYS → ~1,130 lifetime views, +3 subscribers

Video title: quit
Goodbye!
